Module Overview

This module covers everything you need to know about parsing and ingesting data for RAG systems, from basic text files to complex PDFs and databases. We'll use LangChain v0.3 and explore each technique with practical examples.

-Table of Contents

-Introduction to Data Ingestion

-Text Files (.txt)

-Markdown Files (.md)

-PDF Documents

-Microsoft Word Documents

-CSV and Excel Files

-JSON and Structured Data

-Web Scraping

-Databases (SQL)

-Audio and Video Transcripts

-Advanced Techniques

-Best Practices

### Introduction to Data Ingestion


In [3]:
import os
from typing import List, Dict, Any
import pandas as pd


In [4]:
from langchain_core.documents import Document
from langchain.text_splitter import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)
print("Set up Completed !")

Set up Completed !


#### Understanding document Structure in LangChain


In [5]:
## create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Tanya Soni",
        "date_created":"2025-01-01",
        "custom_field":"any_value"
    }
)
print("Document Structure")
print(f"Content:{doc.page_content}")
print(f"Content:{doc.metadata}")

# Why metadata matters:
print("\n📑 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


Document Structure
Content:This is the main text content that will be embedded and searched
Content:{'source': 'example.txt', 'page': 1, 'author': 'Tanya Soni', 'date_created': '2025-01-01', 'custom_field': 'any_value'}

📑 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


## Reading a Text File

In [6]:
## create a simple txt file
import os
os.makedirs("data/text_files",exist_ok=True)

In [7]:
sample_texts={
    "data/text_files/python_intro.txt":"""🔹 What is Python?

Python is a high-level, interpreted, general-purpose programming language.

Created by Guido van Rossum in 1991.

Known for being simple, readable, and beginner-friendly.

🔹 Why Python?

Easy to learn (syntax close to English).

Cross-platform (works on Windows, macOS, Linux).

Huge community and libraries for web development, data science, AI, ML, automation, scripting, etc.

Used by companies like Google, Netflix, Instagram, and NASA.

🔹 Features of Python

Simple & Readable → Great for beginners.

Interpreted → Runs line by line (no need to compile).

Dynamic Typing → No need to declare variable types.

Object-Oriented → Supports classes and objects.

Extensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.

Open Source → Free to use.
    """,
    "data/text_files/ML_intro.txt": """ 🔹 What is Machine Learning?

Machine Learning is a branch of Artificial Intelligence (AI) that allows computers to learn from data and improve automatically without being explicitly programmed.

👉 Instead of giving rules, we give the machine data + examples, and it learns patterns to make predictions or decisions.

🔹 Why Machine Learning?

Used in spam detection (Gmail filters emails).

Recommendation systems (Netflix, Amazon, YouTube).

Self-driving cars (Tesla).

Voice assistants (Siri, Alexa).

Healthcare (disease prediction).

🔹 Types of Machine Learning
1. Supervised Learning

We provide input data (X) and output labels (Y).

Model learns the mapping between them.

Example:

Predicting house price (X = area, location; Y = price).

Email classification (spam / not spam).

🔹 Algorithms: Linear Regression, Decision Trees, Random Forest, Support Vector Machines (SVM).

2. Unsupervised Learning

Data has no labels.

Model tries to find patterns or groups in data.

Example:

Customer segmentation (grouping users by buying behavior).

Market basket analysis (Amazon: “Customers who bought X also bought Y”).

🔹 Algorithms: K-Means, Hierarchical Clustering, PCA (Dimensionality Reduction).

3. Reinforcement Learning (RL)

Agent learns by interacting with an environment and getting rewards/penalties.

Example:

Training robots to walk.

Playing chess or Go (AlphaGo by Google).

Self-driving cars adjusting speed and turns.

🔹 Key Steps in an ML Project

Data Collection – Gather raw data.

Data Preprocessing – Clean and prepare data (handling missing values, scaling, encoding).

Feature Engineering – Selecting important variables.

Model Training – Choosing and training algorithms.

Model Evaluation – Checking accuracy, precision, recall, etc.

Deployment – Using the model in real-world applications (e.g., web app)."""
}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8")as f:
        f.write(content)
print("sample text files created")        

sample text files created


### TextLoader- Read single File

In [8]:
from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

## loading a single file
loader=TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
document=loader.load()
print(type(document))
print(document)
print(f"Loaded {len(document)} document")
print(f"Content preview: {document[0].page_content[:100]}...")
print(f"Metadata: {document[0].metadata}")

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='🔹 What is Python?\n\nPython is a high-level, interpreted, general-purpose programming language.\n\nCreated by Guido van Rossum in 1991.\n\nKnown for being simple, readable, and beginner-friendly.\n\n🔹 Why Python?\n\nEasy to learn (syntax close to English).\n\nCross-platform (works on Windows, macOS, Linux).\n\nHuge community and libraries for web development, data science, AI, ML, automation, scripting, etc.\n\nUsed by companies like Google, Netflix, Instagram, and NASA.\n\n🔹 Features of Python\n\nSimple & Readable → Great for beginners.\n\nInterpreted → Runs line by line (no need to compile).\n\nDynamic Typing → No need to declare variable types.\n\nObject-Oriented → Supports classes and objects.\n\nExtensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.\n\nOpen Source → Free to use.\n    ')]
Loaded 1 document
Content preview: 🔹 What is Python?

Python is a high-level, interpre

### DirectoryLoader- Multiple Text Files

In [9]:
from langchain_community.document_loaders import DirectoryLoader

## load all text files from directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## pattern to match files
    loader_cls=TextLoader, ## loader class to use
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)

documents=dir_loader.load()
print(f"Loaded {len(documents)} documents")
for i,doc in enumerate(documents):
    print(f"\n Document {i+1}:")
    print(f"Source: {doc.metadata['source']}")
    print(f" Length: {len(doc.page_content)} Characters")

print("\n 📂 DirectoryLoader Characteristics:")
print("\n ✅ Advantages:")
print(" - Loads multiple files at once")
print(" - Supports glob patterns")
print(" - Progress tracking")
print(" - Recursive directory scanning")

print("\n ❌ Disadvantages:")
print(" - All files must be same type")
print(" - Limited error handling per file")
print(" - Can be memory intensive for large directories")


100%|██████████| 2/2 [00:00<00:00, 126.59it/s]

Loaded 2 documents

 Document 1:
Source: data\text_files\ML_intro.txt
 Length: 1839 Characters

 Document 2:
Source: data\text_files\python_intro.txt
 Length: 783 Characters

 📂 DirectoryLoader Characteristics:

 ✅ Advantages:
 - Loads multiple files at once
 - Supports glob patterns
 - Progress tracking
 - Recursive directory scanning

 ❌ Disadvantages:
 - All files must be same type
 - Limited error handling per file
 - Can be memory intensive for large directories


### Text Splitting Strategies

In [10]:
### Different text splitting strategies
from langchain.text_splitter import(
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)
print(document)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='🔹 What is Python?\n\nPython is a high-level, interpreted, general-purpose programming language.\n\nCreated by Guido van Rossum in 1991.\n\nKnown for being simple, readable, and beginner-friendly.\n\n🔹 Why Python?\n\nEasy to learn (syntax close to English).\n\nCross-platform (works on Windows, macOS, Linux).\n\nHuge community and libraries for web development, data science, AI, ML, automation, scripting, etc.\n\nUsed by companies like Google, Netflix, Instagram, and NASA.\n\n🔹 Features of Python\n\nSimple & Readable → Great for beginners.\n\nInterpreted → Runs line by line (no need to compile).\n\nDynamic Typing → No need to declare variable types.\n\nObject-Oriented → Supports classes and objects.\n\nExtensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.\n\nOpen Source → Free to use.\n    ')]


In [11]:
### 1. Character Text Splitter
text=document[0].page_content
text

'🔹 What is Python?\n\nPython is a high-level, interpreted, general-purpose programming language.\n\nCreated by Guido van Rossum in 1991.\n\nKnown for being simple, readable, and beginner-friendly.\n\n🔹 Why Python?\n\nEasy to learn (syntax close to English).\n\nCross-platform (works on Windows, macOS, Linux).\n\nHuge community and libraries for web development, data science, AI, ML, automation, scripting, etc.\n\nUsed by companies like Google, Netflix, Instagram, and NASA.\n\n🔹 Features of Python\n\nSimple & Readable → Great for beginners.\n\nInterpreted → Runs line by line (no need to compile).\n\nDynamic Typing → No need to declare variable types.\n\nObject-Oriented → Supports classes and objects.\n\nExtensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.\n\nOpen Source → Free to use.\n    '

In [19]:
### 1. Character Text Splitter
print("CHARACTER TEXT SPLITTER")
char_splitter=CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}....")

CHARACTER TEXT SPLITTER
Created 5 chunks
First chunk: 🔹 What is Python?
Python is a high-level, interpreted, general-purpose programming language.
Created....


In [16]:
print(char_chunks[0])
print("------------")
print(char_chunks[1])
print("------------")
print(char_chunks[2])
print("------------")
print(char_chunks[3])
print("------------")
print(char_chunks[4])


🔹 What is Python?
Python is a high-level, interpreted, general-purpose programming language.
Created by Guido van Rossum in 1991.
Known for being simple, readable, and beginner-friendly.
🔹 Why Python?
------------
🔹 Why Python?
Easy to learn (syntax close to English).
Cross-platform (works on Windows, macOS, Linux).
------------
Huge community and libraries for web development, data science, AI, ML, automation, scripting, etc.
Used by companies like Google, Netflix, Instagram, and NASA.
🔹 Features of Python
------------
🔹 Features of Python
Simple & Readable → Great for beginners.
Interpreted → Runs line by line (no need to compile).
Dynamic Typing → No need to declare variable types.
------------
Object-Oriented → Supports classes and objects.
Extensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.
Open Source → Free to use.


In [22]:
### 1. Recursive Character Splitter
print("Recursive Character Splitter")
recursive_splitter=RecursiveCharacterTextSplitter(
    separators=["\n"],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks=recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}....")

Recursive Character Splitter
Created 5 chunks
First chunk: 🔹 What is Python?

Python is a high-level, interpreted, general-purpose programming language.

Creat....


In [23]:
print(recursive_chunks[0])
print("------------")
print(recursive_chunks[1])
print("------------")
print(recursive_chunks[2])
print("------------")
print(recursive_chunks[3])
print("------------")
print(recursive_chunks[4])

🔹 What is Python?

Python is a high-level, interpreted, general-purpose programming language.

Created by Guido van Rossum in 1991.

Known for being simple, readable, and beginner-friendly.
------------
🔹 Why Python?

Easy to learn (syntax close to English).

Cross-platform (works on Windows, macOS, Linux).
------------
Huge community and libraries for web development, data science, AI, ML, automation, scripting, etc.

Used by companies like Google, Netflix, Instagram, and NASA.

🔹 Features of Python
------------
Simple & Readable → Great for beginners.

Interpreted → Runs line by line (no need to compile).

Dynamic Typing → No need to declare variable types.

Object-Oriented → Supports classes and objects.
------------
Extensive Libraries → NumPy, Pandas, Flask, Django, TensorFlow, etc.

Open Source → Free to use.


In [24]:
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],   # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len,
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks)):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}:'{chunks[i+1]}")

    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2:'two and it is also quite long. This is sentence three which is even longer than

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3:'is even longer than the others. This is sentence four. This is sentence five.

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4:'is sentence five. This is sentence six.

Chunk 4: 'is sentence five. This is sentence six.'


IndexError: list index out of range

In [25]:
### 3. Token Based Splitting
print("Token Text Splitter")
token_splitter=TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
token_chunks=token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"first chunk: {token_chunks[0][:100]}...")



Token Text Splitter
Created 6 chunks
first chunk: 🔹 What is Python?

Python is a high-level, interpreted, general-purpose programming language.

Creat...


In [26]:

print("\n📊 Text Splitting Methods Comparison:")

print("\nCharacterTextSplitter:")
print("✅ Simple and predictable")
print("✅ Good for structured text")
print("❌ May break mid-sentence")
print("➡️ Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("✅ Respects text structure")
print("✅ Tries multiple separators")
print("✅ Best general-purpose splitter")
print("❌ Slightly more complex")
print("➡️ Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("✅ Respects model token limits")
print("✅ More accurate for embeddings")
print("❌ Slower than character-based")
print("➡️ Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
✅ Simple and predictable
✅ Good for structured text
❌ May break mid-sentence
➡️ Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
✅ Respects text structure
✅ Tries multiple separators
✅ Best general-purpose splitter
❌ Slightly more complex
➡️ Use when: Default choice for most texts

TokenTextSplitter:
✅ Respects model token limits
✅ More accurate for embeddings
❌ Slower than character-based
➡️ Use when: Working with token-limited models
